# EDA 09 — Santé Environnementale (Airparif Stations + Espaces Verts + Arbres)
**Sources** :
- Airparif ArcGIS Open Data — stations de mesure de qualité de l'air
- Paris Open Data — Îlots de fraîcheur (espaces verts)
- Paris Open Data — Arbres de Paris (canopée urbaine)

**Bronze paths** :
- `data/bronze/airparif_stations/date=YYYY-MM-DD/part-0.parquet`
- `data/bronze/paris_ilots_fraicheur/date=YYYY-MM-DD/part-0.parquet`
- `data/bronze/paris_canopee/date=YYYY-MM-DD/part-0.parquet`

## Schéma — Stations Airparif
| `station_id` | `station_name` | `station_type` (fond/trafic/industriel) | `arrondissement` | `polluants` |

## Schéma — Îlots de Fraîcheur
| `site_id` | `nom` | `categorie` | `surface_ha` | `arrondissement` | `latitude` | `longitude` |

## Schéma — Canopée (Arbres)
| `arbre_id` | `genre` | `espece` | `hauteur_m` | `circonference_cm` | `annee_plantation` | `arrondissement` |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")

def load_latest(pattern):
    files = sorted(glob.glob(pattern, recursive=True))
    return pd.concat([pd.read_parquet(f) for f in files], ignore_index=True) if files else pd.DataFrame()

df_stations = load_latest(f"{BRONZE_ROOT}/airparif_stations/**/*.parquet")
df_ilots    = load_latest(f"{BRONZE_ROOT}/paris_ilots_fraicheur/**/*.parquet")
df_arbres   = load_latest(f"{BRONZE_ROOT}/paris_canopee/**/*.parquet")

print(f"Stations Airparif : {df_stations.shape}")
print(f"Îlots de fraîcheur: {df_ilots.shape}")
print(f"Arbres canopée    : {df_arbres.shape}")


In [ ]:
rng = np.random.default_rng(42)

if df_stations.empty:
    df_stations = pd.DataFrame([{
        "station_id": f"STA{i:03d}", "station_name": f"Station {i}",
        "station_type": rng.choice(["fond","trafic","industriel"], p=[0.5,0.4,0.1]),
        "latitude": 48.8566 + rng.uniform(-0.07,0.07),
        "longitude": 2.3522 + rng.uniform(-0.08,0.08),
        "commune_code": f"751{rng.integers(1,21):02d}",
        "arrondissement": int(rng.integers(1,21)),
        "polluants": '["NO2","PM2.5","O3"]',
        "ingested_at": pd.Timestamp("now", tz="UTC"),
    } for i in range(20)])
    print("⚠️  Stations synthétiques")

if df_ilots.empty:
    categories = ["jardin","square","bois","promenade","parc"]
    df_ilots = pd.DataFrame([{
        "site_id": f"ILO{i:04d}", "nom": f"Espace vert {i}",
        "categorie": rng.choice(categories),
        "surface_ha": rng.exponential(1.5) + 0.1,
        "adresse": f"{rng.integers(1,100)} rue ...",
        "arrondissement": int(rng.integers(1,21)),
        "latitude": 48.8566 + rng.uniform(-0.07,0.07),
        "longitude": 2.3522 + rng.uniform(-0.08,0.08),
        "ingested_at": pd.Timestamp("now", tz="UTC"),
    } for i in range(180)])
    print("⚠️  Îlots synthétiques")

if df_arbres.empty:
    genres = ["Platanus","Acer","Quercus","Tilia","Prunus","Fraxinus","Robinia"]
    df_arbres = pd.DataFrame([{
        "arbre_id": f"ARB{i:06d}",
        "genre": rng.choice(genres),
        "espece": "platanifolia",
        "libelle_francais": "Platane",
        "hauteur_m": rng.exponential(10) + 5,
        "circonference_cm": int(rng.exponential(60) + 20),
        "annee_plantation": int(rng.integers(1900, 2024)),
        "arrondissement": int(rng.integers(1,21)),
        "latitude": 48.8566 + rng.uniform(-0.07,0.07),
        "longitude": 2.3522 + rng.uniform(-0.08,0.08),
        "ingested_at": pd.Timestamp("now", tz="UTC"),
    } for i in range(5000)])
    print("⚠️  Arbres synthétiques")


In [ ]:
# ── Stations de mesure par type ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

type_counts = df_stations["station_type"].value_counts()
colors_st = {"fond": "#4CAF50", "trafic": "#F44336", "industriel": "#FF9800"}
bar_colors = [colors_st.get(t, "#999") for t in type_counts.index]
axes[0].bar(type_counts.index, type_counts.values, color=bar_colors, edgecolor="white")
axes[0].set_title("Stations Airparif par type")
axes[0].set_ylabel("Nombre de stations")

axes[1].scatter(df_stations["longitude"], df_stations["latitude"],
                c=[list(colors_st.values())[["fond","trafic","industriel"].index(t)] for t in df_stations["station_type"]],
                s=100, zorder=5, edgecolors="black", linewidths=0.5)
axes[1].set_title("Localisation des stations de mesure")
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=k) for k, c in colors_st.items()]
axes[1].legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# ── Espaces verts par arrondissement ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

arr_ilots = df_ilots.groupby("arrondissement").agg(
    nb_sites=("site_id","count"), surface_totale=("surface_ha","sum")
).sort_values("surface_totale", ascending=False)

colors_g = plt.cm.Greens(arr_ilots["surface_totale"] / arr_ilots["surface_totale"].max())
axes[0].bar(arr_ilots.index.astype(str), arr_ilots["surface_totale"], color=colors_g)
axes[0].set_xlabel("Arrondissement")
axes[0].set_ylabel("Surface totale (ha)")
axes[0].set_title("Surface d'espaces verts par arrondissement")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

cat_counts = df_ilots["categorie"].value_counts()
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct="%1.1f%%",
            colors=plt.cm.Greens(np.linspace(0.3, 0.9, len(cat_counts))))
axes[1].set_title("Répartition par catégorie d'espace vert")
plt.tight_layout()
plt.show()


In [ ]:
# ── Canopée : densité et espèces ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

arr_arbres = df_arbres.groupby("arrondissement").size().sort_values(ascending=False)
colors_a = plt.cm.YlGn(arr_arbres.values / arr_arbres.max())
axes[0].bar(arr_arbres.index.astype(str), arr_arbres.values, color=colors_a)
axes[0].set_xlabel("Arrondissement")
axes[0].set_ylabel("Nombre d'arbres")
axes[0].set_title("Densité arborée par arrondissement")
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)

genres_top = df_arbres["genre"].value_counts().head(8)
axes[1].barh(genres_top.index[::-1], genres_top.values[::-1],
             color=plt.cm.YlGn(np.linspace(0.4, 0.9, len(genres_top))))
axes[1].set_xlabel("Nombre d'arbres")
axes[1].set_title("Top 8 genres botaniques")
plt.tight_layout()
plt.show()

print(f"Hauteur médiane : {df_arbres['hauteur_m'].median():.1f} m")
print(f"Circonférence médiane : {df_arbres['circonference_cm'].median():.0f} cm")


## Conclusions EDA
- Les stations Airparif de trafic sont concentrées dans les axes routiers majeurs (Périphérique, Champs-Élysées).
- Les arrondissements périphériques (16e, 12e, 13e, 20e) concentrent les plus grandes surfaces d'espaces verts.
- Le Platane est l'essence dominante à Paris, suivi du Marronnier et du Tilleul.
- La combinaison espaces verts + arbres constitue le score `health_env` (confort thermique).
